# 7. Beam and pointing: the original question

The project started from this question (`research/meerkat-scope.md`):

> Can MeerKAT science observations constrain small physical pointing and
> primary-beam corrections, selected according to their distinguishability
> from uncertain sky emission, while limiting distortion of faint emission
> and source spectra?

It returned to it when the flagging experiment (notebook 6) closed as null.
This notebook walks through what exists so far:

1. the forward model, on the Rust toy fixture, with the JAX reference path
   checked against the Rust baseline (**live**);
2. the smooth pointing solver (**live**, JAX automatic differentiation);
3. the observability audits: which pointing modes the data can see at all
   once sky and gains are free (**live** on small cases; the katbeam audit
   needs a long JAX compile and runs only with `RANG_NOTEBOOK_SLOW=1`);
4. task B1 on real MeerKAT products: the measured reference beam (B1-0) and
   the state-of-the-art pointing fit (B1-1), read from the frozen archives,
   with the fit reproduced **live** on the public fixture;
5. what comes next, B1-2 and B1-3, which are **proposed**: their protocols
   are frozen, no result is reported here.

Everything before section 4 is simulation on an eight-antenna toy; none of it
is a statement about real MeerKAT data.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores
os.environ.setdefault("JAX_PLATFORMS", "cpu")  # the toy is small; no accelerator probe

import pandas as pd
from IPython.display import Markdown, display
from rang.notebook import data, science

print("JAX", science.enable_x64(), "with 64-bit floats")  # before any JAX array

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

CACHE = data.default_cache_dir()  # choose your own with RANG_NOTEBOOK_CACHE
SLOW = science.switch("RANG_NOTEBOOK_SLOW")
HOLOGRAPHY = science.switch("RANG_NOTEBOOK_HOLOGRAPHY")
print("cache:", CACHE)
print(
    "slow JAX audits:",
    "on" if SLOW else "off",
    "| holography maps:",
    "on" if HOLOGRAPHY else "off",
)

## 1. The forward model

For antennas $p, q$, time $t$ and frequency $\nu$, with point components $j$
at direction cosines $(l_j, m_j)$, the visibility is a direct Fourier sum
weighted by the two antennas' voltage beams:

$$V_{pq}(t,\nu)=\sum_j I_j(\nu)\,E_p\bigl(R_t\,s_j-\delta_p(t),\nu\bigr)\,E_q\bigl(R_t\,s_j-\delta_q(t),\nu\bigr)\,e^{-2\pi i\,\nu/c\,(u l_j+v m_j+w(n_j-1))}.$$

$R_t$ rotates sky coordinates $s_j$ into the antenna's beam frame (the
parallactic angle: MeerKAT's alt-az mounts turn the beam on the sky), and
$\delta_p(t)$ is antenna $p$'s pointing offset. A pointing error moves the
beam, so off-axis sources brighten or dim, differently on each baseline.

The **Rust baseline** (`rang-toy`, in the extension) builds a deterministic
toy: eight approximate MeerKAT core positions, 24 times over six hours, four
channels, four bright point sources. The **JAX reference path**
(`rang.pointing`) predicts the same visibilities and differentiates them with
`jax.jacfwd`. The two must agree:

In [ ]:
toy = science.toy_fixture()  # every fifth row of the complete fixture
pd.Series(science.toy_summary(toy), name="toy fixture").to_frame()

In [ ]:
check = science.jax_against_rust(toy)
display(science.as_text(check, "JAX vs Rust"))
display(
    Markdown(
        f"The JAX visibilities match Rust's to {check['max |V_jax − V_rust| (Jy)']:.1e} Jy "
        f"(peak {check['max |V| (Jy)']:.2f} Jy) and the automatic Jacobian matches Rust's "
        f"analytic one to {check['max |J_jax − J_rust| (Jy/arcmin)']:.1e} Jy/arcmin."
    )
)

### The Rust experiment: beam errors and calibration models

`rang.run_experiment` runs the Rust baseline experiment of
`research/first-results.md`: random pointing errors on the toy, a faint
extended source, and six calibration models, from "fixed beam, no pointing
fit" to "joint sky + pointing + shared beam width". One seed, live, with the
correct beam and with a true beam 2 % wider than the model:

In [ ]:
matched, settings = science.rust_experiment_table()
wide, _ = science.rust_experiment_table(beam_error=0.02)
print("settings:", settings)
display(Markdown("**correct beam width** (live, one seed)"))
display(matched)
display(Markdown("**true beam 2 % wider** (live, one seed)"))
wide

The archived campaign repeats this over twenty paired seeds (archive,
`research/results/beam-width-campaign.json`). Read the flux column against
the truth: a small beam-width error biases the faint source's flux unless
the width is fitted, and protecting pointing modes adaptively did not beat
plain joint fitting.

In [ ]:
campaign, truth = science.beam_width_campaign()
print("true flux (mJy):", truth)
campaign

## 2. The smooth pointing solver (JAX)

Each antenna's two offsets are natural cubic splines in time through a few
knots; `rang.pointing.solve_pointing` fits the knot values to the
visibilities by trust-region least squares with the exact Jacobian from
`jax.jacfwd`, a curvature penalty and a weak knot prior. Below: sinusoidal
truth plus a constant offset per antenna, 1 mJy noise, every fourth interior
time held out to score interpolation. Sky and beam are known exactly here,
which is the easy case.

In [ ]:
demo = science.smooth_pointing_demo(toy)
comparison = pd.DataFrame(
    {
        "live": demo["scores"],
        "archive (smooth-pointing.json)": science.smooth_pointing_archive(),
    }
)
display(comparison)
science.trajectory_figure(demo)

A converged fit with full numerical rank is not a certificate: the solver
reports the rank of its data Jacobian, but full rank says nothing about
whether a *physical* ambiguity was broken by the data or by the prior. The
next section asks that question directly.

## 3. Observability: what can the data see at all?

**The Gaussian gauge.** With identical circular beams, a pointing trajectory
shared by every antenna, $\delta_p(t)=R_t\,a$, can be traded exactly against a
change of each source's brightness: every visibility stays the same
(`research/beam-rotation.md`, `research/gaussian-gauge.md`). No solver can
measure such a mode. An elliptical beam that rotates on the sky breaks the
symmetry; one that does not rotate does not.

`rang.observability.sky_locked_information` linearises the model at zero
pointing, profiles out every source/channel flux (and, optionally, gains),
and reports how many shared pointing modes survive and their local
Cramér–Rao bounds. **Live** on the toy, 1 mJy noise:

In [ ]:
symmetry = science.symmetry_audit(toy)
symmetry

Free gains make it harder, and thinning the data can remove a mode entirely.
**Live**: frequency-dependent gains for every antenna, time and channel, with
and without free differential (zero-mean per time) pointing, on the thinned
fixture and the complete one; beside it the archive
(`research/results/gain-observability.json`).

In [ ]:
complete = science.toy_fixture(full=True)
live = science.gain_audit(toy, complete)
archive = science.gain_audit_archive()
display(Markdown("**live**"))
display(live)
display(Markdown("**archive**"))
display(archive)
shared = live.index.intersection(archive.index)
gap = (
    live.loc[shared, "σ axis 1 (arcsec)"] - archive.loc[shared, "σ axis 1 (arcsec)"]
).abs()
print("largest |live − archive| of σ axis 1 (arcsec):", gap.max())

**Chromatic beam shape** (katbeam, the MeerKAT co-polar model fitted to
earlier SARAO holography). The archive ablates the model's squint, its
chromatic axis ratio and its profile, with gains free per antenna, time and
channel, free differential pointing and the other common temporal modes
free. Where the information comes from: chromatic ellipticity, and without
beam rotation nothing survives.

In [ ]:
science.katbeam_audit_archive()

In [ ]:
if SLOW:
    display(Markdown("**live** (katbeam H, native and without rotation)"))
    display(science.katbeam_audit_live(complete))
else:
    print(science.skip_line("RANG_NOTEBOOK_SLOW", "about 30 s of JAX compilation"))

**Where does the information come from?** The information budget
(`research/information-budget.md`) splits the pointing information into what
the visibilities give alone and what uncertain external flux priors add. For
a Gaussian beam the data-only part is zero: any finite σ there is borrowed
from the priors. These are local bounds on a toy, not a MeerKAT forecast
(archive):

In [ ]:
science.information_budget_archive()

## 4. Task B1: the question on real MeerKAT products

The toy results say *where* pointing information could come from; they are
only as good as the beam model. Task B1 (`research/b1-beam-pointing.md`)
froze its protocol before any code and fixed two things first:

- **B1-0, the reference beam**: SARAO's 2023 L-band holography release
  (de Villiers 2023, DOI 10.48479/wdb0-h061, licence CC BY-NC 4.0; only
  derived statistics are committed, the maps stay in ignored outputs);
- **B1-1, the baselines**: the Gaussian pointing-offset fit SARAO runs on
  reference-pointing scans (as the SKA SDP pipeline re-implements it), and
  physical pointing self-calibration.

In [ ]:
pd.DataFrame(science.B1_TASKS, columns=["task", "what", "status"]).set_index("task")

### The unit: the signal of a 30″ offset (live)

B1-0 measures every beam error against the change a 30″ pointing offset
makes, $|\nabla I_b|\cdot 30''$, where $I_b$ is the Stokes I power beam. On
the half-power ring (white) that signal is largest. Here it is for katbeam,
computed live from the public `katbeam` package:

In [ ]:
fig, ring = science.katbeam_signal_figure()
display(pd.Series(ring, name="katbeam, 1284 MHz").to_frame())
fig

### B1-0: the reference beam (archive)

The product passed its three frozen gates (reader and orientation against
SARAO's own metrics file, frequency interpolation, spatial interpolation):

In [ ]:
b1 = science.b1_product()
print("product:", b1["product"]["citation"])
print("licence:", b1["product"]["licence"], "| all gates pass:", b1["gates_pass"])
science.b1_gates(b1)

Against it, the two models rang used before (medians over the product
channels in 900–1650 MHz; |ΔI| as a fraction of the on-axis response):

In [ ]:
comparison = science.b1_comparison(b1)
display(comparison)
s = b1["summary"]
display(
    Markdown(
        f"On the half-power ring katbeam errs by **{s['katbeam']['ring_max_over_30arcsec_signal_median']:.2f}×** "
        f"the signal of a 30″ offset (median channel; range "
        f"{s['katbeam']['ring_max_over_30arcsec_signal_range'][0]:.2f}–"
        f"{s['katbeam']['ring_max_over_30arcsec_signal_range'][1]:.2f}) and EIDOS by "
        f"**{s['eidos']['ring_max_over_30arcsec_signal_median']:.2f}×** "
        f"({s['eidos']['ring_max_over_30arcsec_signal_range'][0]:.2f}–"
        f"{s['eidos']['ring_max_over_30arcsec_signal_range'][1]:.2f}). The measured first "
        f"sidelobe varies from {100 * s['holography']['first_sidelobe_level_range'][0]:.2f} % to "
        f"{100 * s['holography']['first_sidelobe_level_range'][1]:.2f} % across the band. "
        "Neither model replaces the measured beam at the 30″ level everywhere."
    )
)
science.b1_per_channel_figure(b1)

In [ ]:
spread = b1["antenna_spread"]
display(
    Markdown(
        f"Antennas differ from the array average by "
        f"{100 * spread['rms_main_lobe_median_below_1284mhz']:.2f} % RMS in the main lobe below "
        f"1284 MHz and {100 * spread['rms_main_lobe_median_above_1284mhz']:.2f} % above "
        "(median over channels), which matters for per-antenna pointing at the 30″ level."
    )
)
science.b1_antenna_spread_figure(b1)

### B1-1 (i): the state-of-the-art fit, reproduced live

SARAO measures pointing by fitting a Gaussian beam to each antenna's gain
amplitude at five commanded offsets around a calibrator. The public fixture
*is* such an observation: scans 1, 4, 5 and 8 at 1° throws, scan 9 on
source. Rang ports the SKA SDP pipeline's default path
(`examples/b1_baselines.py`, BSD-3-Clause notices kept); its frozen gate G1a
is to reproduce the pipeline's own `reference_offsets.txt`. **Live**, from
the fixture extracted under your cache directory:

In [ ]:
baseline = science.b1_baseline_live(CACHE)
if baseline is None:
    print("skipped: examples/b1_baselines.py is not available outside a checkout")
else:
    display(science.as_text(baseline["g1a"], "gate G1a"))
    est = baseline["estimate"]
    display(est)
    display(
        Markdown(
            f"Every offset lies within {est['|offset| / σ (max of the axes)'].max():.1f} σ of zero, "
            f"and the formal σ runs from {est[['σ xEl', 'σ El']].min().min():.1f}′ to "
            f"{est[['σ xEl', 'σ El']].max().max():.1f}′: on its own test data the fit does not "
            "resolve the pointing at the sub-arcminute level that matters."
        )
    )

**Gate G1b: how biased is the fit against the measured beam?** B1-1
simulated the fixture's five pointings with the holography beam, injected
known offsets and refitted (200 noise seeds). That archive
(`b1-baselines.json`) is not yet committed under `research/results/`; the
cell shows it when it is, or when the ignored run directory is present, and
prints a pointer otherwise.

In [ ]:
g1b = science.b1_baselines_archive()
if g1b is None:
    print(
        "G1b archive not available here: see 'B1-1 (i): the baseline' in "
        "research/b1-beam-pointing.md"
    )
else:
    g1b_record, g1b_path = g1b
    print("archive:", g1b_path.name, "from", g1b_path.parent.name)
    display(science.g1b_table(g1b_record))
    display(science.g1b_figure(g1b_record))

### The holography maps (local only)

The beam maps are SARAO data under CC BY-NC 4.0 and are never committed or
published here. With `RANG_NOTEBOOK_HOLOGRAPHY=1` and the B1-0 product on
disk (`RANG_HOLOGRAPHY_PRODUCT`, or the run directory under `outputs/`), the
cell below draws the array-average Stokes I beam and katbeam's difference
from it.

In [ ]:
product = science.find_holography()
if not HOLOGRAPHY:
    print(science.skip_line("RANG_NOTEBOOK_HOLOGRAPHY"))
elif product is None:
    print(science.skip_line("RANG_NOTEBOOK_HOLOGRAPHY", "no B1-0 product found"))
else:
    display(science.holography_figure(product))

## 5. What comes next: B1-2 and B1-3 (proposed)

Both are specified in the frozen protocol so that their metrics cannot be
tuned after the fact. **Neither has a reported result here.**

**B1-2, identifiability.** The rank and Fisher machinery above, extended
from the toy to real data: beam-derivative directions (per-antenna pointing,
array-common width, ellipticity and squint) computed with the B1-0 beam, on
the LUNO calibrators and on the ESO137 target field, with gains and sky
fluxes profiled out. Its frozen decision rule, quoted from
`rang.notebook.science.B1_2_DECISION_RULE`:

In [ ]:
display(Markdown("> " + science.B1_2_DECISION_RULE))

The protocol states its expectation honestly: on a calibrator the flux sits
at the pointing centre, where the beam is flat, so pointing enters at second
order and is absorbed by the gains; a science field with sources across the
main lobe is where first-order signatures can appear.

**B1-3, pointing self-calibration and rang's method at matched cost**,
against both B1-1 baselines, on four frozen metrics:

In [ ]:
for k, metric in enumerate(science.B1_3_METRICS, 1):
    print(f"{k}. {metric}")
display(
    Markdown("> " + science.B1_3_DECISION_RULE + "\n>\n> " + science.B1_3_MATCHED_COST)
)

---

What this notebook does **not** show: anything about pointing on LUNO or
ESO137 data, the identifiability of beam directions on real data, or a
pointing self-calibration result. Those are B1-2 and B1-3. As in notebook 6,
a null there would be a result, recorded as such.